In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import time

from tqdm import tqdm

import numpy as np
import pandas as pd

import hyperopt.hp as hp

from sklearn.model_selection import train_test_split

from catboost import CatBoostClassifier

from sklift.datasets import fetch_x5
from sklift.models import ClassTransformation, TwoModels
from sklift.metrics import qini_auc_score, uplift_at_k

from causalml.inference.tree import UpliftTreeClassifier as UpliftTreeClassifierCM

from upninja.dml.uplift_tree_dml import UpliftTreeRegressorDML
from upninja.tune.selection import UpliftTune

import matplotlib.pyplot as plt
import seaborn as sns

Failed to import duecredit due to No module named 'duecredit'


In [3]:
SEED = 8

In [4]:
%%time

dataset = fetch_x5()
dataset.data.keys()

CPU times: user 24.6 s, sys: 4.54 s, total: 29.2 s
Wall time: 31.4 s


dict_keys(['clients', 'train', 'purchases'])

In [5]:
%%time

print(f'Dataset type: {type(dataset)}\n')
print(f'Dataset features shape: {dataset.data['clients'].shape}')
print(f'Dataset features shape: {dataset.data['train'].shape}')
print(f'Dataset target shape: {dataset.target.shape}')
print(f'Dataset treatment shape: {dataset.treatment.shape}')

Dataset type: <class 'sklearn.utils._bunch.Bunch'>

Dataset features shape: (400162, 5)
Dataset features shape: (200039, 1)
Dataset target shape: (200039,)
Dataset treatment shape: (200039,)
CPU times: user 81 μs, sys: 78 μs, total: 159 μs
Wall time: 164 μs


In [6]:
%%time

# Извлечение данных
df_clients = dataset.data['clients'].set_index('client_id')
df_train = pd.concat([dataset.data['train'], dataset.treatment , dataset.target], axis=1).set_index('client_id')
indices_test = pd.Index(set(df_clients.index) - set(df_train.index))

# Извлечение признаков
df_features = df_clients.copy()
df_features['first_issue_time'] = \
    (pd.to_datetime(df_features['first_issue_date'])
     - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df_features['first_redeem_time'] = \
    (pd.to_datetime(df_features['first_redeem_date'])
     - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df_features['issue_redeem_delay'] = df_features['first_redeem_time'] \
    - df_features['first_issue_time']
df_features = df_features.drop(['first_issue_date', 'first_redeem_date'], axis=1)

indices_learn, indices_valid = train_test_split(df_train.index, test_size=0.3, random_state=SEED)

CPU times: user 152 ms, sys: 23.8 ms, total: 176 ms
Wall time: 176 ms


In [7]:
%%time

X_train = df_features.loc[indices_learn, :]
y_train = df_train.loc[indices_learn, 'target']
treat_train = df_train.loc[indices_learn, 'treatment_flg']

X_val = df_features.loc[indices_valid, :]
y_val = df_train.loc[indices_valid, 'target']
treat_val =  df_train.loc[indices_valid, 'treatment_flg']

X_train_full = df_features.loc[df_train.index, :]
y_train_full = df_train.loc[:, 'target']
treat_train_full = df_train.loc[:, 'treatment_flg']

X_test = df_features.loc[indices_test, :]

X_train['gender'] = X_train['gender'].map({'F': 0, 'U': -1, 'M': 1})
X_val['gender'] = X_val['gender'].map({'F': 0, 'U': -1, 'M': 1})
X_test['gender'] = X_test['gender'].map({'F': 0, 'U': -1, 'M': 1})

X_train.fillna(-1.0, inplace=True)
X_val.fillna(-1.0, inplace=True)
X_test.fillna(-1.0, inplace=True)

cat_features = ['gender']

CPU times: user 168 ms, sys: 4.77 ms, total: 173 ms
Wall time: 172 ms


# ✅ Train models

## ⭐ Class Transformation

In [8]:
%%time

cb_params = cb_hp_space = {
    'iterations': hp.uniformint('iterations', 100, 2500),
    'depth': hp.uniformint('depth', 2, 10),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1, 10),
    'verbose': False,
    'thread_count': -1,
    'random_state': SEED
}

cb_tune = UpliftTune(
    base_model_class=CatBoostClassifier,
    uplift_model_class=ClassTransformation,
    data=X_train,
    target=y_train,
    treatment=treat_train,
    space=cb_params,
    verbose=True,
    max_evals=10
)

t_time_start = time.process_time_ns()
cb_tuned = cb_tune.tune()['best_params']
t_time_end = time.process_time_ns()

100%|███████| 10/10 [02:37<00:00, 15.79s/trial, best loss: -0.04954288228464684]
Optimization completed. Best score: 0.0495
Best parameters: {'depth': 2, 'iterations': 2166, 'l2_leaf_reg': 6.474242459540229, 'learning_rate': 0.06708005741019658, 'random_state': 8, 'thread_count': -1, 'verbose': False}
CPU times: user 21min 2s, sys: 2min 35s, total: 23min 38s
Wall time: 2min 37s


In [9]:
%%time

ct = ClassTransformation(CatBoostClassifier(**cb_tuned))
f_time_start = time.process_time_ns()
ct = ct.fit(X_train, y_train, treat_train, estimator_fit_params={'cat_features': cat_features})
f_time_end = time.process_time_ns()

ct_uplift = ct.predict(X_val)

CPU times: user 3min, sys: 10.3 s, total: 3min 10s
Wall time: 21.6 s


## ⭐ DML Tree

In [10]:
%%time

space = {
    'min_samples': hp.uniformint('min_samples', 20, 500),
    'max_depth': hp.uniformint('max_depth', 3, 20),
    'random_state': SEED
}

dml_tune = UpliftTune(
    uplift_model_class=UpliftTreeRegressorDML,
    data=X_train,
    target=y_train,
    treatment=treat_train,
    space=space,
    verbose=True,
    max_evals=10
)

t_time_start = time.process_time_ns()
dml_tuned = dml_tune.tune()['best_params']
t_time_end = time.process_time_ns()

100%|███████| 10/10 [00:08<00:00,  1.12trial/s, best loss: -0.05889112202442145]
Optimization completed. Best score: 0.0589
Best parameters: {'max_depth': 14, 'min_samples': 411, 'random_state': 8}
CPU times: user 8.85 s, sys: 90.7 ms, total: 8.94 s
Wall time: 8.94 s


In [11]:
%%time

dml = UpliftTreeRegressorDML(**dml_tuned)

f_time_start = time.process_time_ns()
dml = dml.fit(X_train, y_train, treat_train)
f_time_end = time.process_time_ns()

dml_uplift = dml.predict(X_val)

CPU times: user 550 ms, sys: 3.86 ms, total: 554 ms
Wall time: 553 ms


# ✅ Case

Оценим эффективность назначения коммуникаций при помощи uplift-моделей.

In [20]:
def build_business_case_long(
    base_df,
    purchases_df,
    uplift_col,
    campaign_date=None,
    quantiles=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
):
    base_df = base_df.copy()
    purchases_df = purchases_df.copy()

    purchases_df['transaction_datetime'] = pd.to_datetime(
        purchases_df['transaction_datetime']
    )

    if campaign_date is None:
        campaign_date = purchases_df['transaction_datetime'].min()
    else:
        campaign_date = pd.to_datetime(campaign_date)

    purchases_df['days_since_campaign'] = (
        purchases_df['transaction_datetime'] - campaign_date
    ).dt.days

    purchases_df = purchases_df[purchases_df['days_since_campaign'] >= 0].copy()

    horizons = {
        '1w': 7,
        '2w': 14,
        '3w': 21,
    }

    rows = []

    base_sorted = (
        base_df
        .sort_values(by=uplift_col, ascending=False)
        .reset_index(drop=True)
    )

    total_n = len(base_sorted)

    for q in quantiles:
        cutoff = int(total_n * q)

        top_clients = base_sorted.iloc[:cutoff].copy()
        top_client_ids = top_clients['client_id']

        for horizon_name, max_days in horizons.items():

            horizon_purchases = purchases_df[
                (purchases_df['client_id'].isin(top_client_ids)) &
                (purchases_df['days_since_campaign'] <= max_days)
            ].copy()

            purchase_agg = (
                horizon_purchases
                .groupby('client_id')
                .agg(purchase_sum_horizon=('purchase_sum', 'sum'))
                .reset_index()
            )

            horizon_df = top_clients.merge(
                purchase_agg,
                on='client_id',
                how='left'
            )

            horizon_df['purchase_sum_horizon'] = (
                horizon_df['purchase_sum_horizon'].fillna(0)
            )

            horizon_df['target_horizon'] = (
                horizon_df['purchase_sum_horizon'] > 0
            ).astype(int)

            treat = horizon_df[horizon_df['treatment_flg'] == 1]
            control = horizon_df[horizon_df['treatment_flg'] == 0]

            conv_treat = treat['target_horizon'].mean()
            conv_control = control['target_horizon'].mean()
            uplift_conv = conv_treat - conv_control

            avg_check_treat = treat['purchase_sum_horizon'].mean()
            avg_check_control = control['purchase_sum_horizon'].mean()

            op_per_client = avg_check_treat - avg_check_control - sms_cost
            total_increment = op_per_client * len(horizon_df)

            rows.append({
                'Модель': uplift_col,
                'Горизонт': horizon_name,
                'Доля базы': f'Top {int(q * 100)}%',
                'Доля базы sort': int(q * 100),
                'Кол-во наблюдений': len(horizon_df),

                'Конверсия в КГ': conv_control,
                'Конверсия в ТГ': conv_treat,
                'Инкремент конверсии': uplift_conv,

                'Средний чек в КГ': avg_check_control,
                'Средний чек в ТГ': avg_check_treat,

                'Инкремент ОП на клиента, руб': op_per_client,
                'Total increment, руб': total_increment,
            })

    return pd.DataFrame(rows)

In [21]:
def make_burger_king_table(long_df):
    long_df = long_df.copy()

    horizon_order = ['1w', '2w', '3w']

    horizon_titles = {
        '1w': '1 неделя',
        '2w': '2 неделя',
        '3w': '3 неделя',
    }

    long_df['conv_control_fmt'] = (
        (long_df['Конверсия в КГ'] * 100)
        .round(1)
        .astype(str)
        .str.replace('.', ',', regex=False)
        + '%'
    )

    long_df['conv_treat_fmt'] = (
        (long_df['Конверсия в ТГ'] * 100)
        .round(1)
        .astype(str)
        .str.replace('.', ',', regex=False)
        + '% '
        + long_df['Инкремент конверсии'].apply(
            lambda x: f"(+{x * 100:.1f}%)" if x >= 0 else f"({x * 100:.1f}%)"
        ).str.replace('.', ',', regex=False)
    )

    long_df['op_fmt'] = (
        long_df['Инкремент ОП на клиента, руб']
        .round(1)
    )

    long_df['total_increment_fmt'] = (
        long_df['Total increment, руб']
        .round(0)
        .astype(int)
    )

    final = (
        long_df[['Доля базы', 'Доля базы sort', 'Кол-во наблюдений']]
        .drop_duplicates()
        .copy()
    )

    final = (
        final
        .sort_values('Доля базы sort')
        .drop(columns='Доля базы sort')
        .reset_index(drop=True)
    )

    for h in horizon_order:
        h_df = long_df[long_df['Горизонт'] == h].copy()

        h_df = h_df[[
            'Доля базы',
            'conv_control_fmt',
            'conv_treat_fmt',
            'op_fmt',
            'total_increment_fmt'
        ]].rename(columns={
            'conv_control_fmt': f'{horizon_titles[h]} | Конверсия в КГ',
            'conv_treat_fmt': f'{horizon_titles[h]} | Конверсия в ТГ',
            'op_fmt': f'{horizon_titles[h]} | Инкремент ОП на клиента, руб',
            'total_increment_fmt': f'{horizon_titles[h]} | Total increment, руб',
        })

        final = final.merge(h_df, on='Доля базы', how='left')

    return final

In [14]:
X_val['ct_uplift'] = ct_uplift
X_val['dml_uplift'] = dml_uplift
X_val = pd.concat([X_val, treat_val, y_val], axis=1)

In [15]:
X_val = X_val.merge(dataset.data['purchases'][['client_id', 'purchase_sum', 'transaction_datetime']], how='left', on=['client_id'])

In [18]:
base_val = df_features.loc[indices_valid, :].copy().reset_index()

if 'client_id' not in base_val.columns:
    base_val = base_val.rename(columns={base_val.columns[0]: 'client_id'})

base_val['ct_uplift'] = ct_uplift
base_val['dml_uplift'] = dml_uplift

base_val = base_val.merge(
    treat_val.rename('treatment_flg').reset_index(),
    on='client_id',
    how='left'
)

base_val = base_val.merge(
    y_val.rename('target').reset_index(),
    on='client_id',
    how='left'
)

base_val = base_val.drop_duplicates(subset=['client_id']).copy()

print(base_val.shape)

(60012, 10)


In [19]:
purchases_val = dataset.data['purchases'][[
    'client_id',
    'transaction_datetime',
    'purchase_sum'
]].copy()

purchases_val['transaction_datetime'] = pd.to_datetime(
    purchases_val['transaction_datetime']
)

purchases_val = purchases_val[
    purchases_val['client_id'].isin(base_val['client_id'])
].copy()

print(purchases_val.shape)

(6856118, 3)


In [22]:
dml_long = build_business_case_long(
    base_df=base_val,
    purchases_df=purchases_val,
    uplift_col='dml_uplift',
    campaign_date=None,
    quantiles=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
)

dml_burger_king_table = make_burger_king_table(dml_long)

dml_burger_king_table

,Доля базы,Кол-во наблюдений,1 неделя | Конверсия в КГ,1 неделя | Конверсия в ТГ,"1 неделя | Инкремент ОП на клиента, руб","1 неделя | Total increment, руб",2 неделя | Конверсия в КГ,2 неделя | Конверсия в ТГ,"2 неделя | Инкремент ОП на клиента, руб","2 неделя | Total increment, руб",3 неделя | Конверсия в КГ,3 неделя | Конверсия в ТГ,"3 неделя | Инкремент ОП на клиента, руб","3 неделя | Total increment, руб"
0,Top 10%,6001,"47,0%","42,6% (-4,4%)",-277.2,-1663462,"57,6%","54,4% (-3,2%)",-215.4,-1292664,"65,3%","61,7% (-3,6%)",18.4,110673
1,Top 20%,12002,"54,6%","51,9% (-2,7%)",-74.9,-899379,"67,1%","64,6% (-2,5%)",-28.0,-336398,"74,2%","71,7% (-2,5%)",-225.0,-2700409
2,Top 30%,18003,"54,7%","52,3% (-2,4%)",-93.8,-1689044,"67,8%","65,6% (-2,2%)",62.6,1127636,"75,1%","72,8% (-2,3%)",27.3,491181
3,Top 40%,24004,"55,2%","53,4% (-1,8%)",48.4,1162074,"68,4%","66,9% (-1,6%)",109.1,2618191,"75,8%","74,0% (-1,8%)",156.4,3753502
4,Top 50%,30006,"56,9%","55,7% (-1,2%)",118.1,3544546,"70,3%","69,5% (-0,8%)",185.2,5557813,"77,5%","76,4% (-1,1%)",330.5,9918280
5,Top 60%,36007,"58,0%","57,1% (-0,9%)",98.2,3537223,"71,6%","70,8% (-0,8%)",215.6,7763351,"78,7%","77,6% (-1,1%)",437.9,15767440
6,Top 70%,42008,"57,0%","56,2% (-0,7%)",199.7,8389511,"70,1%","69,8% (-0,3%)",388.4,16316195,"77,0%","76,6% (-0,4%)",667.3,28031062
7,Top 80%,48009,"56,2%","55,7% (-0,4%)",125.3,6015179,"69,3%","69,3% (-0,0%)",312.0,14978018,"76,4%","76,1% (-0,2%)",573.7,27544306
8,Top 90%,54010,"55,7%","55,4% (-0,3%)",73.6,3973878,"68,7%","68,7% (+0,0%)",218.0,11776130,"75,6%","75,5% (-0,1%)",310.1,16748982
9,Top 100%,60012,"53,3%","53,4% (+0,1%)",55.5,3330899,"66,3%","66,6% (+0,3%)",232.0,13921090,"73,5%","73,6% (+0,2%)",322.0,19324203


In [23]:
ct_long = build_business_case_long(
    base_df=base_val,
    purchases_df=purchases_val,
    uplift_col='ct_uplift',
    campaign_date=None,
    quantiles=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8
)

ct_burger_king_table = make_burger_king_table(ct_long)

ct_burger_king_table

,Доля базы,Кол-во наблюдений,1 неделя | Конверсия в КГ,1 неделя | Конверсия в ТГ,"1 неделя | Инкремент ОП на клиента, руб","1 неделя | Total increment, руб",2 неделя | Конверсия в КГ,2 неделя | Конверсия в ТГ,"2 неделя | Инкремент ОП на клиента, руб","2 неделя | Total increment, руб",3 неделя | Конверсия в КГ,3 неделя | Конверсия в ТГ,"3 неделя | Инкремент ОП на клиента, руб","3 неделя | Total increment, руб"
0,Top 10%,6001,"55,0%","48,3% (-6,8%)",-360.2,-2161431,"68,2%","60,7% (-7,6%)",-608.7,-3653032,"74,8%","67,2% (-7,6%)",-646.2,-3878055
1,Top 20%,12002,"56,4%","52,9% (-3,5%)",-197.9,-2374614,"69,9%","66,5% (-3,4%)",-504.0,-6049435,"76,8%","73,4% (-3,4%)",-244.8,-2938059
2,Top 30%,18003,"57,1%","54,9% (-2,2%)",49.3,887559,"70,7%","68,1% (-2,6%)",-76.6,-1378719,"77,5%","75,1% (-2,5%)",276.4,4975560
3,Top 40%,24004,"57,5%","55,9% (-1,6%)",33.2,797034,"71,0%","69,5% (-1,5%)",67.5,1619426,"77,7%","76,2% (-1,5%)",189.4,4546326
4,Top 50%,30006,"57,6%","56,5% (-1,1%)",67.5,2025988,"71,1%","70,0% (-1,0%)",139.4,4183293,"77,9%","76,8% (-1,1%)",379.4,11383088
5,Top 60%,36007,"57,5%","56,8% (-0,8%)",111.9,4030199,"71,0%","70,4% (-0,6%)",296.2,10664096,"78,0%","77,3% (-0,7%)",564.0,20306338
6,Top 70%,42008,"57,5%","56,7% (-0,8%)",92.2,3873059,"70,8%","70,3% (-0,5%)",356.0,14956074,"77,7%","77,1% (-0,6%)",556.3,23369554
7,Top 80%,48009,"57,0%","56,2% (-0,8%)",74.1,3555908,"70,3%","69,9% (-0,4%)",275.1,13205322,"77,3%","76,8% (-0,5%)",383.9,18431866
8,Top 90%,54010,"55,9%","55,3% (-0,6%)",23.3,1258302,"69,2%","68,8% (-0,4%)",174.1,9401753,"76,2%","75,7% (-0,5%)",256.9,13876048
9,Top 100%,60012,"53,3%","53,4% (+0,1%)",55.5,3330899,"66,3%","66,6% (+0,3%)",232.0,13921090,"73,5%","73,6% (+0,2%)",322.0,19324203
